In [0]:
from pyspark.sql import functions as F

claims = spark.table("silver_claims")

checks = {
    "Duplicate claim IDs": claims.count() == claims.select("claim_id").distinct().count(),
    "Invalid claim amounts": claims.filter(F.col("claim_amount").isNull() | (F.col("claim_amount") <= 0)).count() == 0,
    "Null claim IDs": claims.filter(F.col("claim_id").isNull()).count() == 0,
    "Null member IDs": claims.filter(F.col("member_id").isNull()).count() == 0,
    "Null provider IDs": claims.filter(F.col("provider_id").isNull()).count() == 0,
    "Invalid member references": claims.join(spark.table("silver_member").select("member_id"), "member_id", "left_anti").count() == 0,
    "Invalid provider references": claims.join(spark.table("silver_provider").select("provider_id"), "provider_id", "left_anti").count() == 0
    "Blank claim IDs": claims.filter(F.trim(F.col("claim_id")) == "").count() == 0
}

for check, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check}")

if not all(checks.values()):
    raise ValueError("Data quality checks failed!")

print("All data quality checks passed!")